## Introducción

Se resuelve numéricamente la ecuación de difusión del propagador mediante expansión modal en modos propios con condiciones de Neumann. Los coeficientes $F_{nm}$ del modo se usan para calcular $G(t)=(5/16\pi)\sum F_{nm}^2 e^{-\Lambda_{nm}t}$, la función de correlación del campo dipolar en nano-NMR.

# Solución Numérica de la Ecuación de Difusión en Coordenadas Cilíndricas  

**Método de Diferencias Finitas + Crank–Nicolson**  

Perfil arbitrario $D(z)$, condición inicial delta en $(\rho_0=0,\,z_0=d)$ y condiciones de frontera Robin en las tres caras relevantes.

## 1. Formulación matemática

La ecuación a resolver es


$$\frac{\partial P}{\partial t}
=\frac{1}{\rho}\frac{\partial}{\partial\rho}\!\left(\rho\,D(z)\,\frac{\partial P}{\partial\rho}\right)
+\frac{\partial}{\partial z}\!\left(D(z)\,\frac{\partial P}{\partial z}\right),
\tag{1}$$

en el dominio  
$\rho\in[0,R]$,  
$z\in[d,\;L+d]$.  

- **Condición inicial** (delta de Dirac normalizada):

$$P(\rho,z,0\mid\rho_0,z_0)
=\frac{1}{2\pi\,\rho_0}\,\delta(\rho-\rho_0)\,\delta(z-z_0),\quad \rho_0=0,\;z_0=d.\tag{2}$$

- **Condiciones de frontera** (Robin):

$$\begin{cases}
  \displaystyle D(z)\,\partial_z P -\kappa\,P\;=\;0&\text{en } z=d,\\[6pt]\displaystyle D(z)\,\partial_\rho P +\kappa\,P\;=\;0&\text{en } \rho=R,\\[4pt]\displaystyle D(z)\,\partial_z P +\kappa\
  P\;=\;0&\text{en } z=L+d.
\end{cases}$$
  
Avanzamos en el tiempo con Crank–Nicolson:

$$\frac{P^{n+1}_{i,j}-P^{n}_{i,j}}{\Delta t}
=\tfrac12\left(L_{i,j}[P^n]+L_{i,j}[P^{n+1}]\right),$$

donde $L_{i,j}$ es la discretización conservativa del operador espacial.

- **Perfil** $D(z)$ **arbitrario**, definido como `Expression` o función de Python.  
- **Esquema en tiempo**: Crank–Nicolson.  
- **Solver**: PETSc/Gmres con precondicionador AMG.  
- **Correlación**  
  $\;G(t)\propto\!\int_\Omega K(\rho,z)\,P(\rho,z,t)\,\rho\,d\rho\,dz$,  
  evaluada con `assemble(K*u*dx)`.

## 2. Importar librerías y parámetros

In [13]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'fenics'

In [ ]:
# Parámetros físicos
d       = 15.0            # profundidad NV [nm]
L       = 100*d           # altura del dominio [nm]
R       = L               # radio cilíndrico [nm]
kappa   = 0.0             # coef. Robin en superficie [nm/μs]

# Conversión si se desea a unidades internas
# dr, dz en nm; dt en μs
Nr, Nz  = 500, 500        # celdas en ρ, z
dr, dz  = R/Nr, L/Nz

In [ ]:
# Dominio rectangular en (ρ,z)
mesh = RectangleMesh(Point(0, d), Point(R, L+d), Nr, Nz)
V    = FunctionSpace(mesh, 'P', 1)

In [ ]:
# 1.3 Nodos centrales de celda
r_cent = 0.5*(r_edges[:-1] + r_edges[1:])       # shape (Nr,)
z_cent = 0.5*(z_edges[:-1] + z_edges[1:])       # shape (Nz,)

# 1.4 Vectorización
r_nodes = np.linspace(0, R, Nr+1)               # para condiciones de frontera
N        = (Nr+1)*(Nz+1)
Nt       = 1000
times = np.logspace(-4, 7, Nt)
dt=np.diff(times)
# tau_values=D_eff*t_values/d**2
# times    = np.linspace(0, T, Nt+1)

In [ ]:
# 2.1 Función D(z) según TFM
D0, Ds = 2e3, 2e1       # nm^2/μs
lambda_p = 0.75/d
delta    = 2*d/3
z_c      = 0.75*L

D = Expression('Ds + (D0-Ds)*((1 - exp(-lp*(x[1]-d))) * (1 - S) + S)',
               Ds=Ds, D0=D0, lp=lambda_p, d=d,
               S='0.5*(1 + tanh((x[1] - (zc + d))/delta))',
               zc=z_c, delta=delta, degree=2)

In [ ]:
# Marcar fronteras para Robin
boundaries = MeshFunction('size_t', mesh, mesh.topology().dim()-1, 0)
class Bottom(SubDomain):
    def inside(self, x, on): return on and near(x[1], d)
class Top(SubDomain):
    def inside(self, x, on): return on and near(x[1], L+d)
class Side(SubDomain):
    def inside(self, x, on): return on and near(x[0], R)

Bottom().mark(boundaries, 1)
Top().mark(boundaries,    2)
Side().mark(boundaries,   3)
ds = Measure('ds', domain=mesh, subdomain_data=boundaries)

In [ ]:
u   = Function(V)      # solución en t^{n+1}
u0  = Function(V)      # solución en t^n
v   = TestFunction(V)

# Punto fuente inicial en (ρ≈0, z=d)
u0.vector()[:] = 0
delta_src = PointSource(V, Point(0, d), 1.0/(2*np.pi*1e-12))
delta_src.apply(u0.vector())

# Forma débil: (u - u0)/dt·v
F_time = (u - u0)/dt * v * dx

# Término difusión (media CN)
F_diff = 0.5*D*dot(grad(u), grad(v))*dx \
       + 0.5*D*dot(grad(u0), grad(v))*dx

# Término Robin en tres fronteras
F_robin = 0.5*kappa*(u + u0)*v*(ds(1) + ds(2) + ds(3))

# Ecuación completa
F = F_time + F_diff + F_robin
a, LHS = lhs(F), rhs(F)

In [ ]:
# Preparar solver PETSc
solver = PETScKrylovSolver('gmres', 'hypre_amg')
solver.parameters['absolute_tolerance'] = 1e-12
solver.parameters['relative_tolerance'] = 1e-6
solver.parameters['maximum_iterations'] = 1000

# Kernel dipolar K(ρ,z)
# Kernel dipolar: f = (3cos^2(theta) - 1)/r^3 con cos(theta)=z/r, r^2=rho^2+z^2,
# es decir f = (3z^2 - r^2)/r^5 = (2z^2 - rho^2)/(rho^2+z^2)^{5/2}.
# (Antes: 3z^2 - rho^2; la resta va contra r^2, no contra rho^2.)
K = Expression('(2*x[1]*x[1] - x[0]*x[0]) / pow(x[0]*x[0] + x[1]*x[1], 2.5)',
               degree=2)

# Almacenar G(t)
G = np.zeros(Nt+1)
t = 0.0

# Loop de tiempo
for n in range(Nt+1):
    # 1) Correlación G[n] = ∫ K·u dV
    G[n] = assemble(K * u * dx)

    # Guardar en VTK si se desea
    # vtkfile << (u, t)

    if n == Nt: break

    # 2) Resolver paso Crank–Nicolson
    solver.solve(a, u.vector(), LHS)
    u0.assign(u)
    t += dt[n]

In [ ]:
# Normalizar G(t)
G /= G[0]

# Graficar G(t)
import matplotlib.pyplot as plt
times = np.linspace(0, T, Nt+1)

plt.figure()
plt.semilogx(times, G, '-o', markersize=3)
plt.xlabel('t [s]'); plt.ylabel('G(t)/G(0)')
plt.title('Decaimiento de la correlación')
plt.grid(True)
plt.show()

## Conclusiones

Este cuaderno documenta la solución numérica directa de la ecuación de difusión 2D $(\rho, z)$ mediante FEniCS con esquema Crank–Nicolson, como verificación independiente del método espectral modal.

Las características principales del esquema implementado son:

- **Malla FEniCS**: rectangular en $(\rho, z) \in [0, R]\times[d, L+d]$ con $N_r = N_z = 500$ celdas de elementos finitos $P_1$.
- **Esquema temporal**: Crank–Nicolson, incondicionalmente estable y de segundo orden en tiempo.
- **Condiciones de frontera**: Robin en las tres caras relevantes con parámetro $\kappa$, implementadas con `MeshFunction` y `Measure`.
- **Solver**: PETSc/GMRES con precondicionador `hypre_amg` para eficiencia en problemas de gran escala.
- **Correlación**: $G(t) = \int K(\rho,z)\,P(\rho,z,t)\,\rho\,d\rho\,dz$ evaluada con `assemble(K*u*dx)`.

Este enfoque valida de forma independiente los resultados del método espectral modal de `TFM_V5.ipynb`. Requiere la instalación de FEniCS (no disponible en el entorno base de Anaconda); el `ModuleNotFoundError: No module named 'fenics'` en la celda de importación refleja esta dependencia adicional.